### Here we demodulate all scans from the same observation of the calsource in order to recreate the maps of the synthbeam. This is the "all scans" version of `Scan_Demodulation_Src.ipynb`.

# Try to use the demodulation code from qubicpack that demodulates without using a calibration source file, just using a cosine function (see wiki on demodulation)

Some adjustments have to be done on the 2019-04 datasets:
- the three datasets (data, hk and calsource) should have the same t0, so need to synch them with t_xxx = t_xxx + (t_hk_0 - t_xxx_0)

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# Allows the figures to be dynamic
%matplotlib ipympl

In [ ]:
from matplotlib import rc
rc('figure',figsize=(16,8))
rc('font',size=12)
rc('text',usetex=False)

from qubicpack.qubicfp import qubicfp
import qubic.lib.Calibration.Qfiber as ft

import old_qubic_code.demodulation_lib as dl

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
import scipy.ndimage as ndimg
import glob
import string

import iminuit
from iminuit.cost import LeastSquares
import fitsio
from astropy.time import Time
import scipy.signal as scsig
import healpy as hp

import fitsio

from fast_histogram import histogram2d
from scipy.interpolate import RegularGridInterpolator, LinearNDInterpolator

In [ ]:
rc('figure',figsize=(10,7))
# rc('figure',figsize=(5,3.5))

In [ ]:
import qubic.io
from pysimulators import FitsArray
folder_name = "/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_Files/130GHz-2019-04-18/Flat/"
az = FitsArray(folder_name + 'azimuth.fits')
el = FitsArray(folder_name + 'elevation.fits')
file_name = "imgflat_TESNum_{}.fits".format(1)
mymap = FitsArray(folder_name + file_name)

In [ ]:
%%script echo skipping
from matplotlib.widgets import Button

# fig, axs = plt.subplots(1, 4, figsize=(10, 3), width_ratios=(1, 1, 1, 0.05))
fig, axs = plt.subplots(1, 2, figsize=(7, 7), width_ratios=(1, 0.05))

xs = 201
reso = 6

class Index:

    def __init__(self, TES_num, fig, axes, prev_selec, verbose=False):
        self.fig = fig
        self.prev_selec = prev_selec
        self.isok_arr = np.zeros(len(TES_num), dtype=bool)
        self.ind = 0
        self.axs = axes
        self.verbose = verbose
        self.goNext()

    
    def test_ind(self):
        if self.ind >= len(self.isok_arr):
            if self.verbose:
                print("Not good index")
            return False
        elif self.prev_selec[self.ind] == False:
            if self.verbose:
                print("Already rejected TES")
            self.isok_arr[self.ind] = False
            self.goNext()
        else:
            return True
    
    def goNext(self):
        self.ind += 1
        if self.verbose:
            print(self.ind)
        if not self.test_ind():
            return
        if self.verbose:
            print("TES {}".format(self.ind + 1))
        file_name = "imgflat_TESNum_{}.fits".format(self.ind)
        mymap = FitsArray(folder_name + file_name)
        self.axs[0].imshow(mymap)
        plt.subplots_adjust(hspace=0)
        self.fig.suptitle("TES {}".format(self.ind + 1))
        plt.draw()

    def good(self, event):
        if not self.test_ind():
            return
        self.isok_arr[self.ind] = True
        self.goNext()


    def bad(self, event):
        if not self.test_ind():
            return
        self.isok_arr[self.ind] = False
        self.goNext()

allTESNum = np.arange(256) + 1
visibly_ok_arr = np.full_like(allTESNum, True)
test_ifok = np.ones_like(allTESNum, dtype=bool)
callback = Index(allTESNum, fig, axs, visibly_ok_arr, verbose=False)
# callback = Index(allTESNum, fig, axs, np.full_like(visibly_ok_arr, True), verbose=False)
axbad = fig.add_axes([0.7, 0.05, 0.1, 0.075])
axgood = fig.add_axes([0.81, 0.05, 0.1, 0.075])
bgood = Button(axgood, 'Good')
bgood.on_clicked(callback.good)
bbad = Button(axbad, 'Bad')
bbad.on_clicked(callback.bad)

plt.show()

## Dataset

Get the directories corresponding to the days we consider:

In [ ]:
freq_i = 4
caldata_fits = False
freq_map_list = [130, 140, 150, 160, 170]
days_list = [['2019-04-17', '2019-04-18'], ['2019-04-09', '2019-04-10'], ["2019-04-05", "2019-04-06"], ['2019-04-07', '2019-04-08'], ["2019-04-13", "2019-04-14"]] # dowload the data remaining!!
names_list = ["ScanMap_170GHz_Speed_VE4_El", "ScanMap_140GHz_Speed_VE4_El", "ScanMap_Speed_VE4_El", "ScanMap_160GHz_Speed_VE4_El", "ScanMap_170GHz_Speed_VE4_El"] # label error for 130 GHz
bad_cal_time_list = [False, False, False, False, False]

# days = ['2019-04-18']
# days = ['2019-04-17', '2019-04-18']
# data_dir = '/qubic/Data/Calib-TD/'+day+'/'
data_dir = "/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_TOD/"
data_dir_calsrc = "/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_TOD/calsource/"
print(data_dir)
# names = "ScanMap_170GHz_Speed_VE4_El"
# names = "ScanMap_170GHz_Speed_VE4_El_52.81"

days = days_list[freq_i]
names = names_list[freq_i]
bad_cal_time = bad_cal_time_list[freq_i]

dirs = []
fcalsrc = []
for day in days:
    dirs.append(np.sort(glob.glob(data_dir + day + '/*{}*'.format(names))))
    ##### The date format for calsrc has no '-' so we need to change the day to this very format.
    daycalsrc = "".join(str.split(day,'-'))
    fcalsrc.append(np.sort(glob.glob(data_dir_calsrc + "*" + daycalsrc + "*.dat")))
if len(days) > 1: # isinstance(my_array, np.ndarray)
    dirs = np.concatenate(dirs)
    fcalsrc = np.concatenate(fcalsrc)
print("{} directories for TOD.".format(len(dirs)))
print("{} files for calibration source.".format(len(fcalsrc)))
labels = []
for d in dirs:
    bla = str.split(d,'__')
    labels.append(bla[1])
# print(labels)

# print(len(fcalsrc))
# We want an array with all the starting times of the calsource data in unix time
times_calsrc = []
for namef in fcalsrc:
    timef = namef.split("_")[-1]
    times_calsrc.append("{}-{}-{}T{}:{}:{}".format(timef[0:4], timef[4:6], timef[6:8], timef[9:11], timef[11:13], timef[13:15]))
    # print(namef, times_calsrc[-1])
times_calsrc_astropy = Time(times_calsrc, format="isot", scale="utc")
times_calsrc_unix = np.array(times_calsrc_astropy.unix) - 0 * 3600#- 3600*2 # UTC + 2

### Some functions that will be useful

In [ ]:
def read_data_one_TES(thedir, AsicNum, TESNum):
    qfp = qubicfp()
    qfp.read_qubicstudio_dataset(thedir, asic=AsicNum)

    # to get the sums
    data = qfp.timeline(TES=TESNum)
    t_data = qfp.timeaxis(axistype='computertime', TES=TESNum)
    
    # to get the raw data
    # fits = fitsio.FITS("/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_TOD/2019-04-05/2019-04-05_21.20.59__ScanMap_Speed_VE4_El_58.82/Raws/raw-asic1-2019.04.05.212059.fits")
    # print(fits)
    # print(fits[1])
    # table = fits[1].read()
    # print(table)
    # sys.exit()
    # data = qfp.asic_list[AsicNum].hk["ASIC_RAW"]["Raw"][:, TESNum]
    # t_data = qfp.asic_list[AsicNum].hk["ASIC_RAW"]["ComputerDate"]
    # print(np.shape(qfp.asic_list[AsicNum].hk["ASIC_RAW"]["Raw"]))
    # print(np.shape(qfp.asic_list[AsicNum].hk["ASIC_RAW"]["PPS"]))
    # plt.figure()
    # for i in range(5):
    #     plt.plot(t_data, qfp.asic_list[AsicNum].hk["ASIC_RAW"]["Raw"][:, i])
    # plt.plot(t_data, np.mean(qfp.asic_list[AsicNum].hk["ASIC_RAW"]["Raw"], axis=1), c="k")
    # plt.show()
    # print("Asic {}".format(AsicNum))
    # print(np.shape(qfp.asic_list[AsicNum].hk["ASIC_RAW"]["pixelNum"]))
    # sys.exit()

    return t_data, data

In [ ]:
def read_data_one_TES_(thedir, AsicNum, TESNum):
    qfp = qubicfp()
    qfp.read_qubicstudio_dataset(thedir, asic=AsicNum)
    tod = qfp.tod(indextype='TES')
    timeaxis = tod[0]
    todarray = tod[1][TESNum - 1]
    return timeaxis, todarray


In [ ]:
def demodulate_and_rebin(time, data, t_az, az, t_src, src, lowcut, highcut, fmod, nbins, elevation):
    ### Filter Data and Source Signal the same way
    FREQ_SAMPLING = 1./np.mean(time[1:] - time[:-1]) # not all time points are equally distributed
    filt = scsig.butter(5, [lowcut / FREQ_SAMPLING, highcut / FREQ_SAMPLING], btype='bandpass', output='sos')
    # Filter Data and change its sign to be in the same as Src
    new_data = -scsig.sosfilt(filt, data)
    # Interpolate Src on data times and filter it
    new_src = scsig.sosfilt(filt, np.interp(time, t_src, src))

    # Make the product for demodulation
    product = new_data * new_src / np.sum(new_src**2)

    # Smooth it over a period
    ppp = 1./fmod
    size_period = int(FREQ_SAMPLING * ppp)+1
    filter_period = np.ones((size_period,))/size_period
    mov_av = np.convolve(product, filter_period, mode='same') # I didn't check yet if the average shifts the results in az, but might not matter much?
    
    # Rebin this demodulated data as a function of azimuth corrected for elevation
    az_bin, amp_bin, daz, damp, others = ft.profile(np.interp(time, t_az, az),#*np.cos(np.radians(elevation)), 
                                              mov_av, nbins=nbins, cutbad=False,
                                              dispersion=True, plot=False, median=True)
    
    return az_bin, amp_bin, daz, damp



### Loop on all scans for one TES

In [ ]:
method = "rms" # "rms", the other methods don't work for some data sets because it is too hard to
            #synchronize calsource data with science data

time_shifts = [0, 0, 0, 0, 0.5]

AsicNum = 1
TESNum = 93 #93#27
lowcut = 0.1
highcut = 15.
nscans = len(dirs)
nbins = 200
freq_mod = 1 #Hz
period = 1/freq_mod
# test
# delta_t = 2 * 3600 + 0.1 # there seems to be a shift between the two clocks of the order of 100ms (to be checked) + UTC+2
# delta_t = 2 * 3600 # there seems to be a shift between the two clocks of the order of 100ms (to be checked) + UTC+2
allscans_amp = np.zeros((nscans, nbins))
allscans_az = np.zeros((nscans, nbins))
allscans_el = np.zeros((nscans, nbins))
doplot = False
for idir, dir in enumerate(dirs): # we work scan by scan to avoid overflowing the memory
    # if idir not in [65]: # [31, 59, 95] #90
    #     doplot = True
    #     continue
    if freq_i == 2 and idir == 20: # problem at 150 GHz
        allscans_amp[idir] = np.zeros_like(allscans_amp[idir]) + np.nan
        allscans_az[idir] = np.zeros_like(allscans_az[idir]) + np.nan
        allscans_el[idir] = np.zeros_like(allscans_el[idir]) + np.nan
        continue
    print(idir, dir)

    # read the mount data
    t_az, azinit = ft.read_hkintern(dir, thefieldname='Platform-Azimut')
    az = (azinit-2.**15)/(2.**16)*360
    elevation = float(dir[-5:])

    # read the scan data
    t_data, data = read_data_one_TES(dir, AsicNum, TESNum)

    print("size t_data, data = {}, {}".format(np.shape(t_data), np.shape(data))) # 73116, 73116

    # interpolate the mount data to time data
    az_data = np.interp(t_data, t_az, az)

    if method != "rms":
        # read the calsource data
        ttsrc_i = []
        ddsrc_i = []
        for itime, time_cal in enumerate(times_calsrc_unix):
            # print(time_cal)
            if time_cal > np.max(t_data):
                print(time_cal, np.max(t_data))
                break
            try: # in case all the files are already read
                if times_calsrc_unix[itime + 1] > np.min(t_data): # we need only up to the file before the one that starts after the end of our acquisition
                    print("Calfile:", fcalsrc[itime])
                    if caldata_fits:
                        data_calsrc = fitsio.read(fcalsrc[itime])
                        thett = data_calsrc["timestamp"] # the timestamp data are bad for 140 GHz?
                        thedd = data_calsrc["amplitude"]
                    else:
                        data_calsrc = np.genfromtxt(fcalsrc[itime])
                        thett = data_calsrc[:, 0] # the timestamp data are ok in the dat files
                        thedd = data_calsrc[:, 1]
                    # fig, axs = plt.subplots(1, 3)
                    # axs[0].plot(thett)
                    # axs[1].plot(thedd)
                    # axs[2].plot(thett, thedd)
                    # plt.show()
                    # print("mean dtime", np.mean(thett[1:] - thett[:-1]))
                    if bad_cal_time:
                        # the following is needed when the timestamp of the calsource is bad (might actually be due to a pb in the dat to fits conversion)
                        print("!!!Bad timestamps in the calfile!!!")
                        dtime = 0.0005 # on average, the time in seconds between two points measured by the calsource
                        ntimes = len(thett)
                        thett = np.linspace(np.min(thett), np.min(thett) + (ntimes - 1)*dtime, ntimes)
                    ttsrc_i.append(thett)# + delta_t)
                    ddsrc_i.append(thedd)
            except IndexError: # it means we need the last one too (this is unlikely)
                print("We need up to the last calsource file. Check if there is an issue.")
                if caldata_fits:
                    data_calsrc = fitsio.read(fcalsrc[itime])
                    thett = data_calsrc["timestamp"]
                    thedd = data_calsrc["amplitude"]
                else:
                    data_calsrc = np.genfromtxt(fcalsrc[itime])
                    thett = data_calsrc[:, 0]
                    thedd = data_calsrc[:, 1]
                ttsrc_i.append(thett)# + delta_t) # there seems to be a shift between the two clocks of the order of 100ms (to be checked)
                ddsrc_i.append(thedd)

        t_src = np.concatenate(ttsrc_i)
        data_src = np.concatenate(ddsrc_i)

        # test: it seems to be working!
        dt_src = np.min(t_az) - np.min(t_src) # ~2h 0min 7s for 130 GHz dataset
        print("delta t_src = {}".format(dt_src))
        # t_src += dt_src + 0.1 # there is still a slight shift (for the 130 GHz data only?)
        t_src += dt_src + 0.14
    dt_data = np.min(t_az) - np.min(t_data) # ~3s for 130 GHz dataset
    print("delta t_data = {}".format(dt_data))

    # dt_data = 3
    # dt_src = 7204.37789607048
    t_data += dt_data + time_shifts[freq_i]

    # ang_bin, amp_bin, dang, damp = demodulate_and_rebin(t_data, data, t_az, az, t_src, data_src, 
    #                                                     lowcut, highcut, freq_mod, nbins, elevation)

    print(np.shape(t_data))
    # data = data - np.median(data)
    if method == "cal_data":
        az_bin, amp_bin, daz, damp = demodulate_and_rebin(t_data, data, t_az, az, t_src, data_src, 
                                                        lowcut, highcut, freq_mod, nbins, elevation)
    else:
        indata = {"t_data": t_data, "data": data, "t_azel": t_az, "az": az, "el": np.full_like(az, elevation)}

        unbinned, binned = dl.general_demodulate(period, indata, #data, t_src, data_src, t_az, az, 
                                                        lowcut, highcut, #elevation, 
                                                        nbins=nbins, median=True, method=method,
                                                        rebin=True,
                                                        doplot=False, #unbinned=False, 
                                                        renormalize_plot=True)
        az_bin = binned["az"]
        amp_bin = binned["sb"][0] # I work TES by TES
        damp = binned["dsb"][0]

    allscans_amp[idir] = amp_bin
    allscans_az[idir] = az_bin
    allscans_el[idir] = np.full(nbins, elevation)

    if doplot:
        fig, ax = plt.subplots()
        ax.set_title("At elevation={}".format(elevation))
        # ax.plot(np.interp(t_data, t_az, az), data, c="r", label="raw data")
        ax.scatter(np.interp(t_data, t_az, az), data, c="r", label="raw data", s=3)
        ax1 = ax.twinx()
        ax1.plot(az_bin, amp_bin, label="demodulated and rebinned data")
        # ax.set_xlim(-12, -6)
        # ax.set_ylim(0.5e6, 1.5e6)
        plt.legend()
        plt.show()
        fig, ax = plt.subplots()
        ax.plot(t_data, (data - np.mean(data))/np.std(data),'.',label='TES')
        ax.plot(t_az, (az - np.mean(az))/np.std(az),'.',label='Az')
        # plt.plot(t_src, (data_src-np.mean(data_src))/np.std(data_src)/5,',',label='CalSrc')
        if method != "rms":
            ax1 = ax.twinx()
            ax1.plot(t_src, (data_src - np.mean(data_src))/np.std(data_src), '.', c="r", label='CalSrc')
            # ax1.set_ylim([-7, 4])
            max_calsrc = np.max(np.abs((data_src - np.mean(data_src))/np.std(data_src)))
            ax1.set_ylim([-7*max_calsrc, 4*max_calsrc])
        plt.legend()
        # plt.ylim(-3,3)
        ax.set_ylim(-1.5, 2.5)
        # xlim = np.array([78, 86]) + 1.5555201e9
        # plt.xlim(xlim)
        plt.show()
        # azr

In [ ]:
for array in [allscans_amp, allscans_az, allscans_el]:
    if np.sum(~np.isfinite(array)) > 0:
        mask = np.isfinite(array)
        II, JJ = np.meshgrid(np.arange(len(array)), np.arange(len(array[0])), indexing="ij")
        interpolator = LinearNDInterpolator(np.moveaxis([II[mask], JJ[mask]], 0, -1), array[mask], fill_value=0)
        array[:] = interpolator(np.moveaxis([II, JJ], 0, -1)) # use [:] in order to edit the arrays


In [ ]:
print(np.sum(~np.isfinite(allscans_az)))

In [ ]:
def healpix_map(azt, elt, tod, flags=None, flaglimit=0, nside=128, countcut=0, unseen_val=hp.UNSEEN):
    if flags is None:
        flags = np.zeros(len(azt))
    
    ok = flags <= flaglimit 
    return healpix_map_(azt[ok], elt[ok], tod[ok], nside=nside, countcut=countcut, unseen_val=unseen_val)


def healpix_map_(azt, elt, tod, nside=128, countcut=0, unseen_val=hp.UNSEEN):
# def healpix_map_(elt, azt, tod, nside=128, countcut=0, unseen_val=hp.UNSEEN):
    ips = hp.ang2pix(nside, azt, elt, lonlat=True)
    mymap = np.zeros(12*nside**2)
    mapcount = np.zeros(12*nside**2)
    for i in range(len(azt)):
        mymap[ips[i]] += tod[i]
        mapcount[ips[i]] += 1
    unseen = mapcount <= countcut
    mymap[unseen] = unseen_val
    mapcount[unseen] = unseen_val
    mymap[~unseen] = mymap[~unseen] / mapcount[~unseen]
    return mymap, mapcount

In [ ]:
# create a map in az el
map_1, map_count = healpix_map(allscans_az, allscans_el, allscans_amp, nside=340)

In [ ]:
%%script echo use this cell to find position of source
plt.figure()
# plt.plot(allscans_amp[65])
plt.plot(allscans_amp[:, 108])
plt.show()

In [ ]:
# az_source = allscans_az[65, 529] # 1.68406128 # degrees # by hand for now for TES 96
# el_source = allscans_el[65, 529] # 52.02      # degrees # by hand for now for TES 96
i_azs = [106, 108, 108, 108, 108]
i_els = [66, 66, 66, 66, 66]
az_source = allscans_az[i_els[freq_i], i_azs[freq_i]]
el_source = allscans_el[i_els[freq_i], i_azs[freq_i]]
print(az_source, el_source)

In [ ]:
def match_shape(array1, shape_array2): # in the case that array2 has more or the same number of dims
    return np.expand_dims(array1, list(np.arange(len(shape_array2))[1:]))

In [ ]:
def get_perp_vect(point_A, point_B, point_C): # get the vector perpendicular to the plane with these three points of shape (3, ...)
    vec_1 = point_A - match_shape(point_C, point_A.shape)
    vec_2 = point_B - match_shape(point_C, point_B.shape)
    vec_2 = match_shape(vec_2, np.shape(vec_1))
    res_nonorm = np.cross(vec_1, vec_2, axis=0)
    # if len(np.shape(point_A)) == 2:
    #     vec_1 = (point_A - point_C[:, None])
    # else:
    #     vec_1 = (point_A - point_C)
    # if len(np.shape(point_B)) == 2:
    #     vec_2 = (point_B - point_C[:, None])
    #     if len(np.shape(point_A)) == 2:
    #         res_nonorm = np.cross(vec_1, vec_2, axis=0)
    #     else:
    #         res_nonorm = np.cross(vec_1[:, None], vec_2, axis=0)
    # else:
    #     vec_2 = (point_B - point_C)
    #     if len(np.shape(point_A)) != 1:
    #         raise ValueError("Not the right shape for point_A")
    #     res_nonorm = np.cross(vec_1, vec_2, axis=0)
    print("shapes vec_1, vec_2, res_nonorm")
    print(np.shape(vec_1))
    print(np.shape(vec_2))
    print(np.shape(res_nonorm))
    return res_nonorm/np.linalg.norm(res_nonorm, axis=0)

def get_plane(point_A, point_B, point_C, offset=0): # plane equation ax + by + cz + d = 0
    perp_vect = get_perp_vect(point_A, point_B, point_C)
    a, b, c = perp_vect
    d = -np.dot(perp_vect, point_A)
    return np.array([a, b, c, d])

# get two orthogonal vectors on the plane, one pointing at point_ref_end from point_ref_start
def get_vect_plane(plane, point_ref_end, point_ref_start, offset=0): 
    if not np.isclose(np.dot(plane[:3], point_ref_end) + plane[3], 0):
        raise ValueError("The reference end point is not on the plane.")
    if not np.isclose(np.dot(plane[:3], point_ref_start) + plane[3], 0):
        raise ValueError("The reference start point is not on the plane.")
    perp_vect = plane[:3]/np.sqrt(np.sum(plane[:3]**2))
    vec_1 = (point_ref_end - point_ref_start)/np.linalg.norm(point_ref_end - point_ref_start)
    vec_2 = np.cross(perp_vect, vec_1)/np.linalg.norm(np.cross(vec_1, perp_vect))
    return vec_1, vec_2

def spherical2cartesian(rho, theta, phi, coord="spherical", axis="first"): # axis is the axis where the coords for each point will be
    if coord == "horizontal": # theta is azimuth and phi is elevation
        theta_ = theta.copy()
        theta = np.pi/2 - np.radians(phi.copy())
        phi = np.radians(theta_)
        # theta, phi = np.pi/2 - np.radians(phi), np.radians(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    x = rho * np.sin(theta) * np.cos(phi)
    y = rho * np.sin(theta) * np.sin(phi)
    z = rho * np.cos(theta)
    res = np.array([x, y, z])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))
    
def cartesian2spherical(x, y, z, coord="spherical", axis="first"):
    rho = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z/rho)
    phi = np.arctan2(y, x)
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.degrees(phi).copy()
        phi = 90 - np.degrees(theta_)
        # theta, phi = np.degrees(phi), 90 - np.degrees(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    res = np.array([rho, theta, phi])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))
    
def polar2cartesian(r, theta, axis="first"):
    x = r * np.cos(np.radians(theta))
    y = r * np.sin(np.radians(theta))
    res = np.array([x, y])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)

    
def centre_coord_at_0(coord):
    return np.mod(coord - 180, 360) - 180


def mean_2d_bin_data(pos, values, bins): # pos and bins have shape (2, ...)
    # left limit of bins is excluded
    # shape of pos has to be (2, npoints), with npoints in the TOD

    # values_bini = jnp.digitize(pos[0], bins[0]) # right edge of each bin excluded, left edges included
    # values_binj = jnp.digitize(pos[1], bins[1])
    # res, _, _ = jnp.histogram2d(pos[0], pos[1], bins=bins, weights=values)
    res = histogram2d(pos[0], pos[1], range=((bins[0][0], bins[0][-1]), (bins[1][0], bins[1][-1])), bins=(len(bins[0]) - 1, len(bins[1]) - 1), weights=values)
    return res

In [ ]:
# get the intersection between: the great circle perp to constant az = az_pointing and going though pointing; the meridian at az = az_source
def get_intersection(azimuth_pointing, elevation_pointing, azimuth_calsource, sphere_centre=np.array([0, 0, 0]), sphere_radius=1): # à tester
    tilt_az = 1. #1 # degrees # tilt to the pointing horizontal line, to account for a possible tilt of the beam # not perfectly implemented, just a test
    # tilt_el = 1. # degrees # what is this doing?
    elevation_A = np.array([0])
    elevation_B = np.array([90])
    azimuth_calsource = np.array([azimuth_calsource]) # needs to be an array

    # we get vector perpendicular to the pointing horizontal great circle
    perp_vect_pointing = get_perp_vect_horiz_great_circle(azimuth_pointing, elevation_pointing, tilt_az=tilt_az, sphere_centre=sphere_centre, sphere_radius=sphere_radius)

    # we get vector perpendicular to the meridian at source azimuth
    point_A = spherical2cartesian(sphere_radius, azimuth_calsource, elevation_A, coord="horizontal", axis="first")
    point_B = spherical2cartesian(sphere_radius, azimuth_calsource, elevation_B, coord="horizontal", axis="first")
    perp_vec_merid_calsrc = get_perp_vect(point_A, point_B, sphere_centre) # vector perpendicular to meridian at calsource azimuth

    inter_vec = np.cross(perp_vect_pointing, perp_vec_merid_calsrc, axis=0)
    inter_vec /= np.linalg.norm(inter_vec, axis=0)
    inter_cart = inter_vec*sphere_radius + match_shape(sphere_centre, inter_vec.shape)
    x, y, z = inter_cart[0], inter_cart[1], inter_cart[2]
    # print(np.sum(np.isnan(x)), np.sum(np.isnan(y)), np.sum(np.isnan(z)))
    # print(x)
    # print(y)
    # print(z)
    # for coord in [azimuth_pointing, elevation_pointing, x, y, z]:
    #     plt.figure()
    #     plt.imshow(coord)
    #     plt.show()
    return cartesian2spherical(x, y, z, coord="horizontal")


# get the angle between two points (of shape (..., 3))
def dist_angle(vec_A, vec_B):
    cos_angle = np.einsum("...j,...j->...", vec_A, vec_B)/(np.linalg.norm(vec_A, axis=-1) * np.linalg.norm(vec_B, axis=-1))
    mask = np.logical_and(1<cos_angle, cos_angle<1 + 1e-8)
    cos_angle[mask] = 1 # be careful with that, it just seems that it is sometimes a bit higher than 1 because of numerical approximations
    mask = np.logical_and(-1 - 1e-8<cos_angle, cos_angle<-1)
    cos_angle[mask] = -1 # be careful with that, it just seems that it is sometimes a bit higher than 1 because of numerical approximations
    angle = np.arccos(cos_angle) # compute the angle between CA and CB
    sign_angle = np.sign(np.cross(vec_A, vec_B, axis=-1)[..., 2]) # the sign of the z component should give us the sign of the angle
    sign_angle[sign_angle == 0] = 1 # works only if we don't care about the sign
    return sign_angle * angle

# get vector perp to horizontal great circle from az el position
def get_perp_vect_horiz_great_circle(azimuth, elevation, tilt_az=0, sphere_centre=np.array([0, 0, 0]), sphere_radius=1):
    perp_direction = spherical2cartesian(sphere_radius, centre_coord_at_0(azimuth + 180 + tilt_az), 90 - elevation, coord="horizontal", axis="first")
    perp_vect = perp_direction - match_shape(sphere_centre, perp_direction.shape)
    return perp_vect

def get_simple_rotation_matrix(axis, angle):
    one = np.ones_like(angle)
    zero = np.zeros_like(angle)
    if axis == "x":
        R = np.array([[one, zero, zero],
                      [zero, np.cos(angle), -np.sin(angle)],
                      [zero, np.sin(angle), np.cos(angle)]])
    elif axis == "y":
        R = np.array([[np.cos(angle), zero, np.sin(angle)],
                      [zero, one, zero],
                      [-np.sin(angle), zero, np.cos(angle)]])
    elif axis == "z":
        R = np.array([[np.cos(angle), -np.sin(angle), zero],
                      [np.sin(angle), np.cos(angle), zero],
                      [zero, zero, one]])
    else:
        raise ValueError("{} is not a good axis.".format(axis))
    return R


In [ ]:
intersection = get_intersection(allscans_az, allscans_el, az_source, sphere_centre=np.array([0, 0, 0]), sphere_radius=1)
new_allscans_el = intersection[2] 
intersection_az = intersection[1]
print(np.shape(intersection))

In [ ]:
print(allscans_az[89, 197])

In [ ]:
plt.figure()
plt.imshow(new_allscans_el)
plt.show()

plt.figure()
plt.imshow(intersection_az) # constant at source az
plt.show()

plt.figure()
plt.imshow(allscans_az)
plt.show()

plt.figure()
plt.imshow(allscans_el)
plt.show()

In [ ]:
print(len(allscans_amp[np.isnan(new_allscans_el)]))

In [ ]:
%%script echo mistake, skipping
# this is a double mistake, we look for the new amplitudes of pixels at given positions in the old maps but we also created a wrong interpolation, where elevation is increasing on the original amplitudes
# we get a result very close from the one expected because of this double error
def method_1_mistake(allscans_az, allscans_el, allscans_amp):
    def get_pix_cent(pix_min, pix_max, nb_pix):
        return np.linspace(pix_min, pix_max, nb_pix)
    pix_cent = [get_pix_cent(np.min(allscans_el), np.max(allscans_el), len(allscans_amp)), get_pix_cent(np.min(allscans_az), np.max(allscans_az), len(allscans_amp[0]))]
    original_map_interpolator = RegularGridInterpolator(pix_cent, allscans_amp, method='linear', bounds_error=False)
    intersection = get_intersection(allscans_az, allscans_el, az_source, sphere_centre=np.array([0, 0, 0]), sphere_radius=1)
    new_allscans_el = intersection[2]
    return original_map_interpolator(np.moveaxis([new_allscans_el, allscans_az], 0, -1))[::-1]
new_allscans_amp_1 = method_1_mistake(allscans_az, allscans_el, allscans_amp)

In [ ]:
# the method is not perfect, as it doesn't keep the angular distance between peaks
# this is to be improved, namely by modifying the azimuth too!
# on second thoughts, the flat sky approximation should correct for the distortions of the beam with this methods, but only if we don't correct the azimuth?
# the pointings seeing the source with a diagonal side lobe of the same side should be on the same azimuth which is not the same as the pointing seeing the source with an horizontal side lobe!! (see geogebra)
# those diagonal pointings are NOT on elevations symmetrical with respect to the source elevation but have, of course, the same angle with the source (it makes sense that fixed azimuth and fixed angle means different delta_el)
def method_2_not_enough(allscans_az, allscans_el, allscans_amp, az_source):
    intersection = get_intersection(allscans_az, allscans_el, az_source, sphere_centre=np.array([0, 0, 0]), sphere_radius=1)
    new_allscans_el = intersection[2]
    original_map_interpolator = LinearNDInterpolator(np.moveaxis([new_allscans_el.flatten(), allscans_az.flatten()], 0, -1), allscans_amp.flatten(), fill_value=0) # we know the "real" positions of the pixels on the original map and we want to know the amplitudes on a regular grid to create the new map, not the other way around
    return original_map_interpolator(np.moveaxis([allscans_el, allscans_az], 0, -1))
new_allscans_amp_2 = method_2_not_enough(allscans_az, allscans_el, allscans_amp, az_source)

In [ ]:
# still not the best version as only the vertical and horizontal angles with respect to the calsource are respected: diagonals are not well-delt with (even if not visible by eye, weird?)
def method_3_pb_diag(allscans_az, allscans_el, allscans_amp, az_source):
    intersection = get_intersection(allscans_az, allscans_el, az_source, sphere_centre=np.array([0, 0, 0]), sphere_radius=1)
    new_allscans_el = intersection[2]
    intersection_az = intersection[1]
    sphere_radius = 1
    vec_A = spherical2cartesian(sphere_radius, intersection_az, new_allscans_el, coord="horizontal", axis="last") # intersection direction
    vec_B = spherical2cartesian(sphere_radius, allscans_az, allscans_el, coord="horizontal", axis="last") # pointing direction
    angle_ACB = np.degrees(dist_angle(vec_A, vec_B)) # angle between pointing and source meridian
    # corr_fact = np.cos(np.mean([allscans_el, new_allscans_el], axis=0))
    corr_fact = np.cos(np.radians(new_allscans_el))
    # print(np.shape(corr_fact))
    delta_az = angle_ACB/corr_fact - (allscans_az - intersection_az)
    new_allscans_az = allscans_az + delta_az
    # it is pseudo azimuth, that means that a constant new_allscans_az is not a meridian on the sphere but you still need to multiply by np.cos(el) to get the real angle between to points at elevation el
    # the vertical lines in the following maps are constant pseudo azimuth, shouldn't the result look bad? there is something fishy here
    original_map_interpolator2 = LinearNDInterpolator(np.moveaxis([new_allscans_el.flatten(), new_allscans_az.flatten()], 0, -1), allscans_amp.flatten(), fill_value=0) # we know the "real" positions of the pixels on the original map and we want to know the amplitudes on a regular grid to create the new map, not the other way around
    # new_allscans_horiz_angle = corr_fact*(new_allscans_az - az_source)
    # original_map_interpolator2 = LinearNDInterpolator(np.moveaxis([new_allscans_el.flatten(), new_allscans_horiz_angle.flatten()], 0, -1), allscans_amp.flatten(), fill_value=0) # test using the real angle describing the distance between a given pixel and the intersection of the pointing horizontal line with the meridian at az_calsource
    return original_map_interpolator2(np.moveaxis([allscans_el, allscans_az], 0, -1)) # good one
new_allscans_amp_3 = method_3_pb_diag(allscans_az, allscans_el, allscans_amp, az_source)

In [ ]:
# in this method, instead of correcting the azimuth to conserve the angle to the meridian at casource azimuth,
# I want to correct it with respect to the calsource position
# this means that the map has to be in angle with respect to source position and not in pseudo azel?
# I consider the calsource as if it is at the zenith and compute the angular distance and the angle from the source - pointing great circle with
# respect to the horizontal great circle? so that the horizontal great circle is at a given azimuth if the source is the zenith?
def method_4(allscans_az, allscans_el, allscans_amp, az_source, el_source): # not finalized yet
    tilt_az = 1. # degrees
    sphere_radius = 1
    sphere_centre = np.array([0, 0, 0])

    # we get the vector perpendicular to the horizontal great circle at pointing
    perp_vect_pointing = get_perp_vect_horiz_great_circle(allscans_az, allscans_el, tilt_az=tilt_az, sphere_radius=sphere_radius, sphere_centre=sphere_centre)

    # we get vector perpendicular to the great circle going through pointing and calsource
    calsource = spherical2cartesian(sphere_radius, az_source, el_source, coord="horizontal", axis="first")
    pointing = spherical2cartesian(sphere_radius, allscans_az, allscans_el, coord="horizontal", axis="first")
    perp_vec_gc_pointing_calsrc = get_perp_vect(calsource, pointing, sphere_centre)

    # we want the coords to be the last axis
    vec_calsource = np.moveaxis(calsource, 0, -1) # sphere centre is [0, 0, 0]
    vec_pointing = np.moveaxis(pointing, 0, -1)
    # the angle between the pointing and the calsource
    angle_beta = np.abs(np.degrees(dist_angle(vec_calsource, vec_pointing)))

    print(np.argwhere(angle_beta == 0))
    # ar

    print(np.shape(perp_vect_pointing))
    print(np.shape(perp_vec_gc_pointing_calsrc))
    # the angle between the horizontal great circle and the great circle with the pointing and the calsource
    angle_alpha = np.degrees(dist_angle(np.moveaxis(perp_vect_pointing, 0, -1), np.moveaxis(perp_vec_gc_pointing_calsrc, 0, -1)))
    angle_alpha[~np.isfinite(angle_alpha)] = 0 # at the pixel pointing at calsource or if problem for a scan
    angle_beta[~np.isfinite(angle_beta)] = 0 # if problem for a scan

    map_ii, map_jj = np.meshgrid(np.linspace(np.min(angle_beta), np.max(angle_beta), 1000), np.linspace(np.min(angle_alpha), np.max(angle_alpha), 1000))

    # we want to build the map as a sphere seen from the zenith where the calsource is
    # alpha becomes 90° - elevation and beta the azimuth + 90°
    Npix = 500
    # ang_res = (np.max(angle_beta) - np.min(angle_beta))*np.sqrt(2)/Npix
    ang_res = 30/Npix # degrees/pix
    map_ii, map_jj = np.meshgrid(np.arange(Npix), np.arange(Npix), indexing='ij')
    centre = np.array([map_ii[Npix//2, Npix//2], map_jj[Npix//2, Npix//2], 0])
    beta_map = np.sqrt((map_ii-centre[0])**2+(map_jj-centre[1])**2) * ang_res
    vec_pix = np.array([map_ii, map_jj, np.zeros_like(map_jj)]) - match_shape(centre, np.array([map_ii, map_jj, np.zeros_like(map_jj)]).shape)
    vec_zero = np.array([map_ii[Npix//2, -1], map_jj[Npix//2, -1], 0]) - centre # this is not the usual 0 azimuth, but it suits the way I built beta
    alpha_map = -np.degrees(dist_angle(np.moveaxis(vec_pix, 0, -1), np.moveaxis(vec_zero, 0, -1)))
    alpha_map[~np.isfinite(alpha_map)] = 0

    # in order to interpolate around the zenith, I have to first convert alpha beta into cartesian coordinates!
    # I use beta as radius in polar coordinates and alpha as angle, in order to stay in 2D
    cartesian_pos = polar2cartesian(angle_beta, angle_alpha, axis="last")
    cartesian_map = polar2cartesian(beta_map, alpha_map, axis="last")

    plot_0 = angle_alpha
    plot_1 = angle_beta
    # plot_0 = cartesian_pos[..., 0]
    # plot_1 = cartesian_pos[..., 1]
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].set_title("alpha")
    axs[0].imshow(plot_0, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
    axs[1].set_title("beta")
    axs[1].imshow(plot_1, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
    for ax in axs:
        ax.tick_params(
            axis="both",          # changes apply to the x-axis
            which='both',      # both major and minor ticks are affected
            bottom=False,      # ticks along the bottom edge are off
            left=False,         # ticks along the top edge are off
            labelbottom=False,
            labelleft=False) # labels along the bottom edge are off
        ax.set_xlabel("azimuth")
        ax.set_ylabel("elevation")
    plt.tight_layout()
    plt.savefig("azel2alphabeta.pdf")
    plt.show()
    
    # plot_0 = alpha_map
    # plot_1 = beta_map
    plot_0 = cartesian_map[..., 0]
    plot_1 = cartesian_map[..., 1]
    fig, axs = plt.subplots(1, 2)
    axs[0].imshow(plot_0, cmap=plt.cm.gray)
    axs[1].imshow(plot_1, cmap=plt.cm.gray)
    plt.show()

    # here we want 3D in order to rotate and get the new azimuth elevation that I can compare with the original ones
    new_pointing = spherical2cartesian(sphere_radius, alpha_map, 90 - beta_map, coord="horizontal", axis="first") # beta is 90 - elevation!

    ### simpler code
    # rotation_matrix1 = get_simple_rotation_matrix("x", np.radians(90 - el_source)) # rotation y --> z
    # rotation_matrix2 = get_simple_rotation_matrix("z", np.radians(90 - az_source)) # rotation x --> y
    # more rotations but easier to understand
    pre_rotation_matrix = get_simple_rotation_matrix("z", np.radians(90)) # rotation x --> y
    rotation_matrix1 = get_simple_rotation_matrix("y", np.radians(90 - el_source)) # rotation z --> x
    rotation_matrix2 = get_simple_rotation_matrix("z", np.radians(az_source)) # rotation x --> y # azimuth grows from west to east, how do I translate that in x and y? For now it grows from x to y
    new_pointing = np.einsum("ij,jkl->ikl", pre_rotation_matrix, new_pointing) # because the definition of alpha is -90 degrees rotated w.r.t. azimuth at zenith
    
    ### the last axis of both arrays becomes the first one in order to use np.dot
    # los_pos = np.moveaxis(los_pos, -1, 0)
    # rotation_matrix = np.moveaxis(rotation_matrix, -1, 0)
    print("shape new_pointing", np.shape(new_pointing))
    # new_los_pos = np.einsum("il,ikl->kl", los_pos, rotation_matrix) # computation working!
    print("shape rotation_matrix", np.shape(rotation_matrix1), np.shape(rotation_matrix2))

    # plot_0 = new_pointing[0]
    # plot_1 = new_pointing[1]
    # plot_2 = new_pointing[2]
    # fig, axs = plt.subplots(1, 3)
    # axs[0].imshow(plot_0, cmap=plt.cm.gray)
    # axs[1].imshow(plot_1, cmap=plt.cm.gray)
    # axs[2].imshow(plot_2, cmap=plt.cm.gray)
    # plt.show()

    new_pointing = np.einsum("ij,jkl->ikl", rotation_matrix1, new_pointing)

    # plot_0 = new_pointing[0]
    # plot_1 = new_pointing[1]
    # plot_2 = new_pointing[2]
    # fig, axs = plt.subplots(1, 3)
    # axs[0].imshow(plot_0, cmap=plt.cm.gray)
    # axs[1].imshow(plot_1, cmap=plt.cm.gray)
    # axs[2].imshow(plot_2, cmap=plt.cm.gray)
    # plt.show()


    new_pointing = np.einsum("ij,jkl->ikl", rotation_matrix2, new_pointing)

    # plot_0 = new_pointing[0]
    # plot_1 = new_pointing[1]
    # plot_2 = new_pointing[2]
    # fig, axs = plt.subplots(1, 3)
    # axs[0].imshow(plot_0, cmap=plt.cm.gray)
    # axs[1].imshow(plot_1, cmap=plt.cm.gray)
    # axs[2].imshow(plot_2, cmap=plt.cm.gray)
    # plt.show()

    # zae
    print(np.shape(new_pointing))
    _, new_az, new_el = cartesian2spherical(new_pointing[0], new_pointing[1], new_pointing[2], coord="horizontal", axis="first")

    plot_0 = new_az
    plot_1 = new_el
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].set_title("new az")
    axs[0].imshow(plot_0, cmap=plt.cm.gray)
    axs[1].set_title("new el")
    axs[1].imshow(plot_1, cmap=plt.cm.gray)
    for ax in axs:
        ax.tick_params(
            axis="both",          # changes apply to the x-axis
            which='both',      # both major and minor ticks are affected
            bottom=False,      # ticks along the bottom edge are off
            left=False,         # ticks along the top edge are off
            labelbottom=False,
            labelleft=False) # labels along the bottom edge are off
        ax.set_xlabel("azimuth")
        ax.set_ylabel("elevation")
    plt.tight_layout()
    plt.savefig("newazel.pdf")
    plt.show()
    # zar

    # we now operate in angle from the calsource at the zenith
    print(np.sum(np.isnan(allscans_amp)))
    print(np.sum(np.isnan(cartesian_pos)))


    original_map_interpolator4 = LinearNDInterpolator(cartesian_pos.reshape(cartesian_pos.shape[0]*cartesian_pos.shape[1], 2), allscans_amp.flatten(), fill_value=0) # we know the "real" positions of the pixels on the original map and we want to know the amplitudes on a regular grid to create the new map, not the other way around
    return original_map_interpolator4(cartesian_map), new_az, new_el # good one
    # original_map_interpolator4 = LinearNDInterpolator(np.moveaxis([angle_alpha.flatten(), angle_beta.flatten()], 0, -1), allscans_amp.flatten(), fill_value=0) # we know the "real" positions of the pixels on the original map and we want to know the amplitudes on a regular grid to create the new map, not the other way around
    # return original_map_interpolator4(np.moveaxis([alpha_map, beta_map], 0, -1)) # good one
new_allscans_amp_4, new_az, new_el = method_4(allscans_az, allscans_el, allscans_amp, az_source, el_source)

In [ ]:
%%script echo skipping
sphere_radius = 1
vec_A = spherical2cartesian(sphere_radius, intersection_az, new_allscans_el, coord="horizontal", axis="last")
vec_B = spherical2cartesian(sphere_radius, allscans_az, allscans_el, coord="horizontal", axis="last")

angle_ACB = np.degrees(dist_angle(vec_A, vec_B))
# corr_fact = np.cos(np.mean([allscans_el, new_allscans_el], axis=0))
corr_fact = np.cos(np.radians(new_allscans_el))
print(np.shape(corr_fact))
delta_az = angle_ACB/corr_fact - (allscans_az - intersection_az)
new_allscans_az = allscans_az + delta_az
# it is pseudo azimuth, that means that a constant new_allscans_az is not a meridian on the sphere but you still need to multiply by np.cos(el) to get the real angle between to points at elevation el
# the vertical lines in the following maps are constant pseudo azimuth, shouldn't the result look bad? there is something fishy here
new_allscans_horiz_angle = corr_fact*(new_allscans_az - az_source)
# original_map_interpolator2 = LinearNDInterpolator(np.moveaxis([new_allscans_el.flatten(), new_allscans_az.flatten()], 0, -1), allscans_amp.flatten(), fill_value=0) # we know the "real" positions of the pixels on the original map and we want to know the amplitudes on a regular grid to create the new map, not the other way around
original_map_interpolator2 = LinearNDInterpolator(np.moveaxis([new_allscans_el.flatten(), new_allscans_horiz_angle.flatten()], 0, -1), allscans_amp.flatten(), fill_value=0) # test using the real angle describing the distance between a given pixel and the intersection of the pointing horizontal line with the meridian at az_calsource
new_allscans_amp2 = original_map_interpolator2(np.moveaxis([allscans_el, allscans_az], 0, -1)) # good one
# new_allscans_amp2 = original_map_interpolator(np.moveaxis([new_allscans_el, new_allscans_az], 0, -1))[::-1]
# print("allscans_az - intersection_az", allscans_az - intersection_az)
# print("angle_ACB/corr_fact", angle_ACB/corr_fact)
# print("new_allscans_el", new_allscans_el)
# print("corr_fact", corr_fact)
print("delta_az", delta_az)

fig, axs = plt.subplots(1, 5)
axs[0].imshow(corr_fact*(allscans_az - intersection_az)/angle_ACB, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
axs[1].imshow(angle_ACB, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
axs[2].imshow(new_allscans_horiz_angle, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
axs[3].imshow(delta_az, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
axs[4].imshow(new_allscans_az, cmap=plt.cm.gray, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
vmin = 0   # -0.1
vmax = 1e5 # 0.1
multiplier = 1 # -1

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].set_title("no correction")
axs[0].imshow(multiplier*allscans_amp, cmap=plt.cm.gray, vmin=vmin, vmax=vmax, aspect=len(allscans_amp[0])/len(allscans_amp))#, aspect=aspect)#, aspect=len(allscans_amp[0])/len(allscans_amp))
xylabels = [["azimuth", "elevation"]]
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*new_allscans_.az) - np.min(corr_fact*new_allscans_az)) * len(allscans_amp[0])/len(allscans_amp)
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*new_allscans_horiz_angle) - np.min(corr_fact*new_allscans_horiz_angle)) * len(allscans_amp[0])/len(allscans_amp)
axs[1].set_title("corrected")
axs[1].imshow(multiplier*new_allscans_amp_4, cmap=plt.cm.gray, vmin=vmin, vmax=vmax)#, aspect=aspect)#, aspect=len(allscans_amp[0])/len(allscans_amp))
xylabels += [["new azimuth", "new elevation"]]
for i_ax, ax in enumerate(axs):
    ax.tick_params(
        axis="both",          # changes apply to the x-axis
        which='both',      # both major and minor ticks are affected
        bottom=False,      # ticks along the bottom edge are off
        left=False,         # ticks along the top edge are off
        labelbottom=False,
        labelleft=False) # labels along the bottom edge are off
    ax.set_xlabel(xylabels[i_ax][0])
    ax.set_ylabel(xylabels[i_ax][1])
plt.tight_layout()
plt.savefig("comp_nocorr_meth4.pdf")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2)
axs[0].imshow(multiplier*new_allscans_amp_2, cmap=plt.cm.gray, vmin=vmin, vmax=vmax, aspect=len(allscans_amp[0])/len(allscans_amp))#, aspect=aspect)#, aspect=len(allscans_amp[0])/len(allscans_amp))
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*new_allscans_.az) - np.min(corr_fact*new_allscans_az)) * len(allscans_amp[0])/len(allscans_amp)
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*new_allscans_horiz_angle) - np.min(corr_fact*new_allscans_horiz_angle)) * len(allscans_amp[0])/len(allscans_amp)
axs[1].imshow(multiplier*new_allscans_amp_4, cmap=plt.cm.gray, vmin=vmin, vmax=vmax)#, aspect=aspect)#, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2)
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*allscans_az) - np.min(corr_fact*allscans_az)) * len(allscans_amp[0])/len(allscans_amp)
axs[0].imshow(multiplier*new_allscans_amp_2, cmap=plt.cm.gray, vmin=vmin, vmax=vmax, aspect=len(allscans_amp[0])/len(allscans_amp))#, aspect=aspect)#, aspect=len(allscans_amp[0])/len(allscans_amp))
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*new_allscans_.az) - np.min(corr_fact*new_allscans_az)) * len(allscans_amp[0])/len(allscans_amp)
# aspect = (np.max(new_allscans_el) - np.min(new_allscans_el))/(np.max(corr_fact*new_allscans_horiz_angle) - np.min(corr_fact*new_allscans_horiz_angle)) * len(allscans_amp[0])/len(allscans_amp)
axs[1].imshow(multiplier*new_allscans_amp_3, cmap=plt.cm.gray, vmin=vmin, vmax=vmax, aspect=len(allscans_amp[0])/len(allscans_amp))#, aspect=aspect)#, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.imshow(multiplier*allscans_amp, cmap=plt.cm.gray, vmin=vmin, vmax=vmax, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
sys.exit()

In [ ]:
# fitsio.write("test_maps/TES_96.fits", -allscans_amp, clobber=True)
# fitsio.write("test_maps/azimuth.fits", allscans_az, clobber=True)
# fitsio.write("test_maps/elevation.fits", allscans_el, clobber=True)

# fitsio.write("test_maps/corr_azimuth.fits", allscans_az, clobber=True)
# fitsio.write("test_maps/corr_elevation.fits", new_allscans_el, clobber=True)
# fitsio.write("test_maps/corr_TES_96.fits", -new_allscans_amp, clobber=True)
# fitsio.write("test_maps/corr_TES_{}_2.fits".format(TESNum), -new_allscans_amp_2, clobber=True)
# fitsio.write("test_maps/corr_TES_96_3.fits", -new_allscans_amp_3, clobber=True)

add_on = "_" + method + "_" + str(freq_map_list[freq_i]) + "GHz"
fitsio.write("test_maps/corr_azimuth_4{}.fits".format(method), new_az, clobber=True) # azimuth if calsource is at zenith and then rotated at calsource position
fitsio.write("test_maps/corr_elevation_4{}.fits".format(method), new_el, clobber=True) # elevation if calsource is at zenith and then rotated at calsource position
fitsio.write("test_maps/corr_TES_{}_4{}.fits".format(TESNum, add_on), -new_allscans_amp_4, clobber=True)

In [ ]:
%%script echo skipping
# check how the map looks
rot = (np.mean(allscans_az), np.mean(allscans_el))
map_1_flat = hp.gnomview(-map_1, rot=rot, xsize=1000, reso=2, return_projected_map=True, no_plot=True).data
map_1_flat[map_1_flat == hp.UNSEEN] = 0
plt.figure()
plt.imshow(map_1_flat, cmap=plt.cm.gray, vmin=-0.05, vmax=0.05)
plt.show()

In [ ]:
def spherical2cartesian(rho, theta, phi, coord="spherical", axis="first"):
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.pi/2 - np.radians(phi.copy())
        phi = np.radians(theta_)
        # theta, phi = np.pi/2 - np.radians(phi), np.radians(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    x = rho * np.sin(theta) * np.cos(phi)
    y = rho * np.sin(theta) * np.sin(phi)
    z = rho * np.cos(theta)
    res = np.array([x, y, z])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))
    



def cartesian2spherical(x, y, z, coord="spherical", axis="first"):
    rho = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z/rho)
    phi = np.arctan2(y, x)
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.degrees(phi).copy()
        phi = 90 - np.degrees(theta_)
        # theta, phi = np.degrees(phi), 90 - np.degrees(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    res = np.array([rho, theta, phi])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))

In [ ]:
def get_new_azel(azt, elt, azsrc, elsrc): # adapted from get_new_azel_v2 in pipeline_moon_functions.py # 2 rotations to put the source at the right position?
    rho_sphere = 1
    moon_sph = np.radians(np.array([90 - elsrc, azsrc]))
    moon_pos = spherical2cartesian(rho_sphere, moon_sph[0], moon_sph[1])
    los_sph = np.radians(np.array([90 - elt, azt]))
    los_pos = spherical2cartesian(rho_sphere, los_sph[0], los_sph[1])

    ### simpler code
    rotation_matrix1 = get_simple_rotation_matrix("z", np.radians(azsrc)) # rotation x --> y, but I put it at the wrong position later so it's the oppsite. To be improved
    rotation_matrix2 = get_simple_rotation_matrix("y", np.radians(-elsrc)) # rotation z --> x, but idem

    ### the last axis of both arrays becomes the first one in order to use np.dot
    # los_pos = np.moveaxis(los_pos, -1, 0)
    # rotation_matrix = np.moveaxis(rotation_matrix, -1, 0)
    # new_los_pos = np.dot(los_pos, rotation_matrix) # crashes
    # new_los_pos = np.vecmat(los_pos, rotation_matrix) # not yet in numpy 1.26...
    print("shape los_sph", np.shape(los_pos))
    # print("shape rotation_matrix", np.shape(rotation_matrix))
    # new_los_pos = np.einsum("il,ikl->kl", los_pos, rotation_matrix) # computation working!
    print("shape rotation_matrix", np.shape(rotation_matrix1), np.shape(rotation_matrix2))
    new_los_pos = np.einsum("il,ikl->kl", los_pos, rotation_matrix1)
    new_los_pos = np.einsum("il,ikl->kl", new_los_pos, rotation_matrix2)

    ### to see if we get always (0, 0, 0) for the Moon positions
    for i in range(3):
        print(moon_pos[i])
    new_moon_pos = np.einsum("il,ikl->kl", moon_pos, rotation_matrix1)
    print("First rotation")
    for i in range(3):
        print(new_moon_pos[i])
    new_moon_pos = np.einsum("il,ikl->kl", new_moon_pos, rotation_matrix2)
    print("Second rotation")
    for i in range(3):
        print(new_moon_pos[i])
    print(np.allclose(new_moon_pos[1], np.zeros_like(new_moon_pos[1])))
    print(np.allclose(new_moon_pos[2], np.zeros_like(new_moon_pos[2])))
    print(np.shape(new_los_pos))
    x, y, z = new_los_pos
    print(np.shape(x), np.shape(y), np.shape(z))
    res = cartesian2spherical(x, y, z)
    rho, theta, phi = res[..., 0], res[..., 1], res[..., 2]
    if not np.all(np.isclose(rho, 1)):
        raise ValueError("The radius of the sphere is not one.")
    
    newazt = np.degrees(phi)
    newelt = 90 - np.degrees(theta)
    return newazt, newelt
